# A small autoencoder on CPU

Compress 64 pixel values into an eight-dimensional code and reconstruct the image. The model learns from images without using their class labels. We choose an epoch on validation reconstruction loss and report the held-out test loss.

**Prerequisites:** MLPs, training loops, and mean squared error. **Environment:** base requirements plus CPU PyTorch. **Execution:** restart and run all cells. No external dataset, pretrained weights, or GPU is required. This reconstructive autoencoder is not a video generator.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, stratify=y_development, random_state=SEED
)
print('Train / validation / test:', len(y_train), len(y_validation), len(y_test))
X_train, X_validation, X_test = X_train / 16, X_validation / 16, X_test / 16
def image_loader(images, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(images, dtype=torch.float32)),
                      batch_size=64, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))
train_batches = image_loader(X_train, shuffle=True)
validation_batches = image_loader(X_validation)


## Encoder, bottleneck, and decoder

The encoder maps 64 input pixels to eight latent values. The decoder expands them back to 64 pixels. Its final sigmoid matches the normalized `[0,1]` input range. Class labels are kept only for viewing examples after training and never enter the loss.


In [ ]:
class DigitAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 24), nn.ReLU(), nn.Linear(24, 8))
        self.decoder = nn.Sequential(nn.Linear(8, 24), nn.ReLU(), nn.Linear(24, 64), nn.Sigmoid())
    def forward(self, images):
        return self.decoder(self.encoder(images))

torch.manual_seed(SEED)
model = DigitAutoencoder().to(DEVICE)
probe = torch.tensor(X_train[:4])
assert model.encoder(probe).shape == (4, 8)
assert model(probe).shape == probe.shape
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))


## Minimize pixel reconstruction error

Mean squared error averages over images and pixels. The evaluation helper accumulates summed squared errors and divides by the number of pixels, so batches of different sizes are weighted correctly. Retain a copy of the lowest-validation-loss weights; do not use the test images to choose an epoch.


In [ ]:
def reconstruction_loss(model, batches):
    model.eval()
    squared_error, pixels = 0.0, 0
    with torch.no_grad():
        for (images,) in batches:
            images = images.to(DEVICE)
            squared_error += nn.functional.mse_loss(model(images), images, reduction='sum').item()
            pixels += images.numel()
    return squared_error / pixels

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
rows, best_state, best_validation_loss = [], None, float('inf')
for epoch in range(1, 16):
    model.train()
    squared_error, pixels = 0.0, 0
    for (images,) in train_batches:
        images = images.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(images), images)
        loss.backward()
        optimizer.step()
        squared_error += loss.item() * images.numel()
        pixels += images.numel()
    validation_loss = reconstruction_loss(model, validation_batches)
    rows.append({'epoch': epoch, 'train_mse': squared_error / pixels, 'validation_mse': validation_loss})
    if validation_loss < best_validation_loss:
        best_validation_loss, best_state = validation_loss, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(rows)
display(history.tail())
history.plot(x='epoch', y=['train_mse', 'validation_mse'], figsize=(6, 3))
plt.ylabel('Mean squared error per pixel')
plt.show()


## Evaluate and inspect held-out reconstructions

Low reconstruction error alone does not establish useful classification features or realistic image generation. The bottleneck may discard fine details, which is visible by comparing held-out originals and reconstructions. The plot's top row shows originals; the bottom row shows reconstructions.


In [ ]:
test_mse = reconstruction_loss(model, image_loader(X_test))
print('Held-out mean squared error per pixel:', round(test_mse, 6))
with torch.no_grad():
    reconstructed = model(torch.tensor(X_test[:8])).numpy().reshape(-1, 8, 8)
assert np.isfinite(reconstructed).all()
assert ((reconstructed >= 0) & (reconstructed <= 1)).all()
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for index in range(8):
    axes[0, index].imshow(X_test[index].reshape(8, 8), cmap='gray', vmin=0, vmax=1)
    axes[0, index].set_title(f'Label {y_test[index]}')
    axes[1, index].imshow(reconstructed[index], cmap='gray', vmin=0, vmax=1)
    for row in range(2):
        axes[row, index].axis('off')
axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Reconstruction')
plt.tight_layout()
plt.show()


## Practice

Compare bottleneck widths using validation reconstruction loss. For a denoising experiment, create noisy training inputs but retain clean training targets; keep test images untouched until evaluation.

**References:** [PyTorch MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html), [scikit-learn digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html).
